In [1]:
pip install pypdf sentence-transformers faiss-cpu numpy transformers torch

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Hp\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import torch

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected. Will use CPU (slower).")
    print("For GPU support, install PyTorch with CUDA:")
    print("pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")

PyTorch version: 2.12.0+cu130
CUDA available: True
CUDA version: 13.0
GPU count: 1
GPU name: NVIDIA GeForce RTX 3050 Laptop GPU
GPU memory: 4.29 GB


In [3]:
import os
import glob
import pickle
import numpy as np
from typing import List, Dict

DATA_DIR = "data/pdfs"
STORAGE_DIR = "storage"
INDEX_PATH = os.path.join(STORAGE_DIR, "faiss.index")
METADATA_PATH = os.path.join(STORAGE_DIR, "metadata.pkl")

CHUNK_SIZE = 300          # characters per chunk
CHUNK_OVERLAP = 50       # overlap between chunks
EMBEDDING_MODEL = "intfloat/e5-large-v2"
TOP_K = 3                 # retrieve top 3 relevant chunks
LLM_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

In [4]:
pip install pymupdf

Defaulting to user installation because normal site-packages is not writeable
  Using cached pymupdf-1.27.2.3-cp310-abi3-win_amd64.whl.metadata (24 kB)
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   - -------------------------------------- 0.8/19.2 MB 6.2 MB/s eta 0:00:04
   ---- ----------------------------------- 2.4/19.2 MB 7.0 MB/s eta 0:00:03
   ------- -------------------------------- 3.7/19.2 MB 6.8 MB/s eta 0:00:03
   ---------- ----------------------------- 5.2/19.2 MB 7.1 MB/s eta 0:00:02
   -------------- ------------------------- 7.1/19.2 MB 7.4 MB/s eta 0:00:02
   ----------------- ---------------------- 8.7/19.2 MB 7.6 MB/s eta 0:00:02
   -------------------- ------------------- 10.0/19.2 MB 7.4 MB/s eta 0:00:02
   ------------------------ --------------- 11.8/19.2 MB 7.5 MB/s eta 0:00:01
   --------------------------- ------------ 13.4/19.2 MB 7.5 MB/s eta 0:00:01
   ---------


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Hp\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import fitz  # pymupdf

def load_pdf(pdf_path: str) -> List[Dict]:

    pages = []

    doc = fitz.open(pdf_path)

    for page_num, page in enumerate(doc, start=1):

        text = page.get_text("text")

        # Clean whitespace
        text = " ".join(text.split())

        if text:
            pages.append({
                "page": page_num,
                "text": text
            })

    return pages

In [6]:
def split_text(text, chunk_size=500, overlap=75):

    words = text.split()

    chunks = []
    current = []

    current_len = 0

    for word in words:

        if current_len + len(word) + 1 <= chunk_size:
            current.append(word)
            current_len += len(word) + 1

        else:
            chunks.append(" ".join(current))

            overlap_words = current[-15:] if len(current) > 15 else current

            current = overlap_words + [word]
            current_len = len(" ".join(current))

    if current:
        chunks.append(" ".join(current))

    return chunks

In [7]:
def chunk_pages(pages: List[Dict], chunk_size: int, overlap: int, doc_name: str) -> List[Dict]:
    chunks = []
    chunk_id = 0
    for page in pages:
        page_num = page["page"]
        page_text = page["text"]

        page_text = page_text.replace("\n", " ")
        page_text = " ".join(page_text.split())
        split_chunks = split_text(page_text, chunk_size, overlap)
        for chunk_text in split_chunks:
            if chunk_text.strip():
                chunks.append({
                    "chunk_id": chunk_id,
                    "document": doc_name,
                    "page": page_num,
                    "text": chunk_text.strip()
                })
                chunk_id += 1
    return chunks

In [8]:
pip install numpy scipy scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Hp\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [9]:
from sentence_transformers import SentenceTransformer
import torch
import numpy as np

class EmbeddingModel:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.model = SentenceTransformer(
            EMBEDDING_MODEL,
            device=self.device
        )

        print(f"Embedding model running on {self.device}")

    def encode(self, texts):
        embeddings = self.model.encode(
            texts,
            batch_size=64,              # increase if enough VRAM
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        )

        return embeddings.astype(np.float32)

C:\Users\Hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
import faiss

class VectorStore:
    def __init__(self, dimension: int):
        self.dimension = dimension
        self.index = faiss.IndexFlatIP(dimension)
        self.metadata: List[Dict] = []

    def add(self, embeddings: np.ndarray, metadata: List[Dict]):
        if embeddings.shape[0] != len(metadata):
            raise ValueError("Mismatch between embeddings and metadata")
        self.index.add(embeddings)
        self.metadata.extend(metadata)

    def search(self, query_embedding: np.ndarray, k: int):
        scores, indices = self.index.search(query_embedding, k)
        return scores[0], indices[0]

    def save(self):
        os.makedirs(STORAGE_DIR, exist_ok=True)
        faiss.write_index(self.index, INDEX_PATH)
        with open(METADATA_PATH, "wb") as f:
            pickle.dump(self.metadata, f)

    @classmethod
    def load(cls, dimension: int):
        obj = cls(dimension)
        obj.index = faiss.read_index(INDEX_PATH)
        with open(METADATA_PATH, "rb") as f:
            obj.metadata = pickle.load(f)
        return obj

In [11]:
class Retriever:
    def __init__(self, vector_store: VectorStore, embed_model: EmbeddingModel):
        self.vector_store = vector_store
        self.embed_model = embed_model

    def retrieve(self, query: str, top_k: int = TOP_K):
        query_vec = self.embed_model.encode([query])
        scores, indices = self.vector_store.search(query_vec, top_k)
        results = []
        for idx, score in zip(indices, scores):
            if idx != -1:
                chunk_info = self.vector_store.metadata[idx].copy()
                chunk_info["score"] = float(score)
                results.append(chunk_info)
        return results

In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("Loading Hugging Face LLM...")

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME
    
).to(device)



print(f"LLM loaded on {device}")


def build_prompt(query: str, contexts: list[dict]) -> str:

    context_text = "\n\n".join([
        f"[Document: {ctx['document']} | Page {ctx['page']}]\n{ctx['text']}"
        for ctx in contexts
    ])

    prompt = f"""
Context:

{context_text}

Question:
{query}



Answer:
"""

    return prompt


def ask_hf_llm(prompt: str, max_new_tokens: int = 200):

    messages = [
        {
            "role": "system",
            "content":
              """
You are a RAG assistant.
You are a document question answering assistant.

Rules:
1. Answer ONLY from the provided context.
2. Give a short, natural answer.
3. Do NOT copy entire paragraphs.
4. Summarize the information in your own words.
5. If the answer is missing, say exactly:
I cannot find that information in the document.
"""
               
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    chat_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        chat_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=3500
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.5,
            top_p=1.0,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.15,
            no_repeat_ngram_size=4
        )

    # Remove the prompt tokens
    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

Loading Hugging Face LLM...


Loading weights: 100%|██████████| 195/195 [00:00<00:00, 1750.56it/s]


LLM loaded on cuda


In [14]:
def index_all_pdfs():
    all_chunks = []
    pdf_files = glob.glob(os.path.join(DATA_DIR, "*.pdf"))
    if not pdf_files:
        print(f"No PDFs found in {DATA_DIR}. Please upload PDFs to that folder.")
        return None, None

    embed_model = EmbeddingModel()
    for pdf_path in pdf_files:
        print(f"Processing {pdf_path}...")
        pages = load_pdf(pdf_path)
        doc_name = os.path.basename(pdf_path)
        chunks = chunk_pages(pages, CHUNK_SIZE, CHUNK_OVERLAP, doc_name)
        all_chunks.extend(chunks)

    if not all_chunks:
        print("No chunks extracted.")
        return None, None

    print(f"Generated {len(all_chunks)} chunks")
    chunk_texts = [c["text"] for c in all_chunks]
    embeddings = embed_model.encode(chunk_texts)

    dim = embeddings.shape[1]
    vector_store = VectorStore(dim)
    vector_store.add(embeddings, all_chunks)
    vector_store.save()
    print(f"Saved FAISS index with {len(all_chunks)} chunks")
    return vector_store, embed_model

# Build or load index
if os.path.exists(INDEX_PATH) and os.path.exists(METADATA_PATH):
    print("Loading existing index...")
    embed_model = EmbeddingModel()
    dummy_emb = embed_model.encode(["dummy"])
    dim = dummy_emb.shape[1]
    vector_store = VectorStore.load(dim)
    print("Index loaded.")
else:
    print("No existing index found. Building from PDFs...")
    vector_store, embed_model = index_all_pdfs()
    if vector_store is None:
        raise SystemExit("Indexing failed. Check your PDF files and folder.")

retriever = Retriever(vector_store, embed_model)
print("\n✅ Indexing complete. You can now ask questions.")

No existing index found. Building from PDFs...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2003.10it/s]


Embedding model running on cuda
Processing data/pdfs\The-History-of-AI_Avicena.pdf...
Generated 61 chunks


Batches: 100%|██████████| 1/1 [00:26<00:00, 26.91s/it]

Saved FAISS index with 61 chunks

✅ Indexing complete. You can now ask questions.


In [15]:
def ask_question():

    query = input("\n❓ Your question: ").strip()

    if not query:
        return

    contexts = retriever.retrieve(query)

    if not contexts:
        print("No relevant chunks found.")
        return

    contexts = remove_duplicates(contexts)

    print("\n📄 Retrieved sources:\n")

    for ctx in contexts:
        print(
            f"- {ctx['document']} "
            f"(page {ctx['page']}) "
            f"score={ctx['score']:.3f}"
        )
        print(ctx["text"])
        print("-" * 80)

    prompt = build_prompt(query, contexts)

    answer = ask_hf_llm(prompt)

    print("\n🤖 Answer:\n")
    print(answer)

In [16]:
def remove_duplicates(contexts):

    seen = set()
    unique = []

    for ctx in contexts:
        text = ctx["text"].strip()

        if text not in seen:
            unique.append(ctx)
            seen.add(text)

    return unique

In [17]:
ask_question()

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.70it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



📄 Retrieved sources:

- The-History-of-AI_Avicena.pdf (page 1) score=0.827
of modern life. The History of AI Multiple definitions of AI exist, but the one that captures the essence of AI is: "Artificial Intelligence encompasses a collection of sciences, theories, and techniques, including mathematical logic, statistics, probabilities, computational neurobiology, and
--------------------------------------------------------------------------------
- The-History-of-AI_Avicena.pdf (page 4) score=0.827
potential applications of AI across diverse fields such as healthcare, finance, and customer service, where the capacity to understand and respond to natural language is paramount. The 2020s to Present A pivotal milestone in the evolution of artificial intelligence is arguably marked by the
--------------------------------------------------------------------------------
- The-History-of-AI_Avicena.pdf (page 3) score=0.819
and AI was Alan Turing, a mathematician, logician, computer scientist,